# Sweep 12a — Notes Thesis: Core Isolation + Validity Checks

**27 runs ≈ 4–5 hours**

| Group | What | Runs |
|-------|------|------|
| **A** | naked → +notes → +notes+copy → +notes+labs → full | 15 |
| **B** | Leakage test: `remove_med_section` on vs off | 6 |
| **C** | Historical notes: `--use_hist_notes` on vs off | 6 |

**Run order matters:**
- A0 (naked) and A1 (notes-only) are reference baselines for ALL notes experiments including 12b.
- B is the most important validity check in the thesis: if B1 (med section kept) >> B0 (removed), notes are benefiting from seeing the discharge medication list, inflating Δ_notes.

**Backbone for all groups:** hgt_ehr_feat, transformer, no CAMO.  
Groups B/C/D/E/F all use naked+notes (no labs, no copy) so deltas are pure note effects.

In [ ]:
import torch
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
!pip install -q torch_geometric
!pip install -q pyyaml pandas numpy scikit-learn transformers

In [ ]:
import os, sys, glob

if os.path.exists("/kaggle"):
    print("Running on Kaggle")
    os.chdir("/kaggle/working")
    os.system("rm -rf ./data ./src")
    os.makedirs("data/processed", exist_ok=True)
    os.makedirs("data/embeddings", exist_ok=True)

    train_paths = glob.glob("/kaggle/input/**/train.py", recursive=True)
    if not train_paths:
        raise FileNotFoundError("train.py not found in /kaggle/input")
    src_dir = os.path.dirname(train_paths[0])
    os.system(f"cp -r {src_dir} /kaggle/working/src")
    sys.path.append("/kaggle/working/src")

    for find_file in ["cohort_mimic3.pkl", "records_final.pkl"]:
        found = glob.glob(f"/kaggle/input/**/{find_file}", recursive=True)
        if found:
            processed_dir = os.path.dirname(found[0])
            print(f"Found processed dir at: {processed_dir}")
            for root, dirs, files in os.walk(processed_dir):
                rel = os.path.relpath(root, processed_dir)
                target_dir = os.path.join("./data/processed", rel) if rel != "." else "./data/processed"
                os.makedirs(target_dir, exist_ok=True)
                for fname in files:
                    src = os.path.join(root, fname)
                    link = os.path.join(target_dir, fname)
                    if not os.path.exists(link):
                        try:
                            os.symlink(src, link)
                        except OSError:
                            os.system(f"cp '{src}' '{link}'")
            break

    emb_paths = glob.glob("/kaggle/input/**/code_embeddings.pt", recursive=True)
    if not emb_paths:
        raise FileNotFoundError("code_embeddings.pt not found")
    emb_dir = os.path.dirname(emb_paths[0])
    for fpath in glob.glob(f"{emb_dir}/*"):
        fname = os.path.basename(fpath)
        link = f"./data/embeddings/{fname}"
        if not os.path.exists(link):
            os.symlink(fpath, link)

    reg_path = "/kaggle/working/src/model/registry.py"
    if os.path.exists(reg_path):
        with open(reg_path, "r") as rf:
            content = rf.read()
        if "import inspect" not in content:
            old = "        return cls(**kwargs)"
            new = (
                "        import inspect\n"
                "        sig = inspect.signature(cls.__init__)\n"
                "        has_var_keyword = any(p.kind == inspect.Parameter.VAR_KEYWORD "
                "for p in sig.parameters.values())\n"
                "        if has_var_keyword:\n"
                "            filtered_kwargs = kwargs\n"
                "        else:\n"
                "            filtered_kwargs = {k: v for k, v in kwargs.items() if k in sig.parameters}\n"
                "        return cls(**filtered_kwargs)"
            )
            if old in content:
                content = content.replace(old, new)
                with open(reg_path, "w") as wf:
                    wf.write(content)
                print("  registry.py patched")

print("Setup done:", os.getcwd())

In [ ]:
import yaml
from pathlib import Path

BASE_CONFIG = "src/config.yaml"

def make_config(output_path, model_overrides=None, preprocessing_overrides=None, smoke_epochs=None):
    with open(BASE_CONFIG, "r") as f:
        cfg = yaml.safe_load(f)
    if model_overrides:
        for k, v in model_overrides.items():
            cfg["model"][k] = v
    if preprocessing_overrides:
        for k, v in preprocessing_overrides.items():
            cfg["preprocessing"][k] = v
    if smoke_epochs is not None:
        cfg["training"]["epochs"] = smoke_epochs
    with open(output_path, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False)
    return str(output_path)

BACKBONE_MODEL = {
    "graph_encoder_type": "hgt_ehr_feat",
    "hgt_layers": 1,
    "pos_weight_cap": 15.0,
}
BACKBONE_FLAGS = (
    "--device cuda "
    "--visit_level_training "
    "--fusion_strategy film "
)

# Base overrides for note experiments: notes ON, no labs, no copy
NOTES_MODEL_OV = {"copy_mechanism": False, **BACKBONE_MODEL}
NOTES_PREP_OV  = {"lab_dim": 0}  # note_method stays default (clinicalbert_chunk_pool)

SEEDS = [42, 123, 456]
reports_dir = Path("experiment_reports/sweep_12a_notes_core")
results_log = []

print("Config helpers ready.")

## Group B prerequisite — generate no-med-section embeddings

The leakage test requires a second note embedding pkl where the discharge medication section was NOT removed.  
The default `note_embeddings_mimic3.pkl` was generated with `remove_med_section=True`.  
This cell generates `note_embeddings_mimic3_with_meds.pkl` from `notes_text_mimic3.pkl` using the same ClinicalBERT model but skipping the med section filter.

**Skip this cell if `data/processed/note_embeddings_mimic3_with_meds.pkl` already exists.**

In [ ]:
from pathlib import Path

WITH_MEDS_PKL = Path("data/processed/note_embeddings_mimic3_with_meds.pkl")

if WITH_MEDS_PKL.exists():
    print(f"Already exists: {WITH_MEDS_PKL} — skipping generation.")
else:
    import pickle, numpy as np, torch
    from transformers import AutoTokenizer, AutoModel

    print("Loading notes_text_mimic3.pkl ...")
    with open("data/processed/notes_text_mimic3.pkl", "rb") as f:
        notes_text = pickle.load(f)  # expected: dict {hadm_id -> text} or list

    print(f"Loaded {len(notes_text)} notes")
    print(f"Type: {type(notes_text)}, sample key: {next(iter(notes_text)) if isinstance(notes_text, dict) else 'list'}")

    # Load existing reference pkl to get hadm_id ordering
    with open("data/processed/note_embeddings_mimic3.pkl", "rb") as f:
        ref = pickle.load(f)
    hadm_ids = ref["hadm_ids"]
    has_note  = ref["has_note"]
    n_total   = len(hadm_ids)

    print(f"\nLoading ClinicalBERT ...")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()
    print(f"  ClinicalBERT loaded on {device}")

    def encode_text(text, chunk_size=512, overlap=128):
        """Chunk-pool encode text WITHOUT removing medication section."""
        tokens = tokenizer.encode(text, add_special_tokens=False)
        if len(tokens) == 0:
            return np.zeros(768, dtype=np.float32)
        step = chunk_size - overlap
        chunks = [tokens[i:i+chunk_size] for i in range(0, max(1, len(tokens)-overlap), step)]
        chunk_embs = []
        for chunk in chunks:
            if len(chunk) < 10:
                continue
            enc = tokenizer(tokenizer.decode(chunk), return_tensors="pt",
                            max_length=512, truncation=True, padding=True).to(device)
            with torch.no_grad():
                out = model(**enc)
            chunk_embs.append(out.last_hidden_state[:, 0, :].squeeze().cpu().numpy())
        if not chunk_embs:
            return np.zeros(768, dtype=np.float32)
        return np.mean(chunk_embs, axis=0).astype(np.float32)

    print(f"\nEncoding {n_total} admissions (with medication section) ...")
    embeddings = np.zeros((n_total, 768), dtype=np.float32)

    # Diagnostic: show actual pkl structure so we can verify the keys
    if isinstance(notes_text, dict):
        print(f"  notes_text keys: {list(notes_text.keys())}")
        for k, v in notes_text.items():
            try:
                sample = list(v.items())[:2] if isinstance(v, dict) else v[:2]
            except Exception:
                sample = repr(v)[:80]
            print(f"    key={repr(k)} type={type(v).__name__} sample={repr(sample)}")

    if isinstance(notes_text, dict):
        first_key = next(iter(notes_text))
        if isinstance(first_key, int) or (isinstance(first_key, str) and str(first_key).lstrip("-").isdigit()):
            # Standard {hadm_id: text} mapping
            notes_lookup = {int(k): v for k, v in notes_text.items()}
        else:
            # Structured dict e.g. {"hadm_ids": [...], "notes": [...]} or {"hadm_ids": [...], "notes": {id: text}}
            _hids  = notes_text.get("hadm_ids", notes_text.get("hadm_id", []))
            _raw   = notes_text.get("text", notes_text.get("texts",
                     notes_text.get("note",  notes_text.get("notes", []))))
            # _raw may itself be a dict {hadm_id: text} or a parallel list
            if isinstance(_raw, dict):
                notes_lookup = {int(k): v for k, v in _raw.items()}
            else:
                notes_lookup = {int(h): t for h, t in zip(_hids, _raw)}
    else:
        notes_lookup = {int(hadm_ids[i]): notes_text[i] for i in range(len(notes_text))}

    for i, hid in enumerate(hadm_ids):
        text = notes_lookup.get(int(hid), "")
        # Guard: value must be a non-empty string (pkl may store 0/None for missing notes)
        if isinstance(text, str) and len(text.strip()) > 20:
            embeddings[i] = encode_text(text)
        if i % 500 == 0:
            print(f"  {i}/{n_total}")

    out_data = {
        "embeddings": embeddings,
        "has_note":   has_note,
        "hadm_ids":   hadm_ids,
        "method":     "clinicalbert_chunk_pool_with_meds",
    }
    WITH_MEDS_PKL.parent.mkdir(parents=True, exist_ok=True)
    with open(WITH_MEDS_PKL, "wb") as f:
        pickle.dump(out_data, f)
    print(f"\nSaved: {WITH_MEDS_PKL}")
    del model, tokenizer
    torch.cuda.empty_cache()

In [ ]:
# Smoke tests — 1 epoch
import subprocess
from pathlib import Path

Path("smoke_s12a").mkdir(exist_ok=True)

SMOKE = [
    # Group A
    {"name": "A0_naked",
     "model_ov": {"copy_mechanism": False, **BACKBONE_MODEL},
     "prep_ov": {"note_method": "none", "lab_dim": 0}, "flags": ""},
    {"name": "A1_notes",
     "model_ov": NOTES_MODEL_OV, "prep_ov": NOTES_PREP_OV, "flags": ""},
    {"name": "A4_full",
     "model_ov": BACKBONE_MODEL,
     "prep_ov": {"lab_dim": 400},
     "flags": "--lab_pkl data/processed/lab_vectors_200labs.pkl --num_labs 200"},
    # Group B
    {"name": "B1_with_meds",
     "model_ov": NOTES_MODEL_OV, "prep_ov": NOTES_PREP_OV,
     "flags": "--note_pkl data/processed/note_embeddings_mimic3_with_meds.pkl"},
    # Group C
    {"name": "C1_hist_notes",
     "model_ov": NOTES_MODEL_OV, "prep_ov": NOTES_PREP_OV, "flags": "--use_hist_notes"},
]

smoke_results = []
for t in SMOKE:
    cfg_path = f"s12a_smoke_{t['name']}.yaml"
    make_config(cfg_path, model_overrides=t["model_ov"],
                preprocessing_overrides=t["prep_ov"], smoke_epochs=1)
    cmd = (f"python -u src/train.py --config {cfg_path} "
           f"{BACKBONE_FLAGS} {t['flags']} "
           f"--seed 42 --results_dir smoke_s12a/{t['name']}")
    print(f"SMOKE {t['name']}: {cmd}")
    proc = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    tail = (proc.stdout + proc.stderr).strip().split("\n")[-5:]
    for l in tail: print(l)
    status = "PASS" if proc.returncode == 0 else f"FAIL ({proc.returncode})"
    smoke_results.append(f"{status}: {t['name']}")
    print(f">>> {status}\n")

for r in smoke_results: print(r)
if any("FAIL" in r for r in smoke_results):
    raise RuntimeError("Smoke tests failed.")
print("All smoke tests passed.")

## Group A — Core modality sweep (notes axis)

Same structure as 11a but from the notes perspective.
A0 and A1 are reference values for all of 12a and 12b.

In [ ]:
import subprocess, gc, torch

GROUP_A = [
    {
        "name": "A0_naked",
        "model_ov": {"copy_mechanism": False, **BACKBONE_MODEL},
        "prep_ov": {"note_method": "none", "lab_dim": 0},
        "flags": "",
        "note": "Structural floor — no notes, no labs, no copy.",
    },
    {
        "name": "A1_notes_only",
        "model_ov": NOTES_MODEL_OV,
        "prep_ov": NOTES_PREP_OV,
        "flags": "",
        "note": "+ notes only. Clean Δ_notes = A1 - A0.",
    },
    {
        "name": "A2_notes_copy",
        "model_ov": BACKBONE_MODEL,  # copy ON
        "prep_ov": {"lab_dim": 0},
        "flags": "",
        "note": "+ notes + copy (no labs). Does copy amplify note signal?",
    },
    {
        "name": "A3_notes_labs",
        "model_ov": NOTES_MODEL_OV,
        "prep_ov": {"lab_dim": 400},
        "flags": "--lab_pkl data/processed/lab_vectors_200labs.pkl --num_labs 200",
        "note": "+ notes + labs (no copy). Notes+labs combined without copy.",
    },
    {
        "name": "A4_full",
        "model_ov": BACKBONE_MODEL,
        "prep_ov": {"lab_dim": 400},
        "flags": "--lab_pkl data/processed/lab_vectors_200labs.pkl --num_labs 200",
        "note": "Full system: notes + labs + copy.",
    },
]

for cfg in GROUP_A:
    print(f"\n{'#'*70}\n# {cfg['name']} — {cfg['note']}\n{'#'*70}")
    cfg_path = f"s12a_{cfg['name']}.yaml"
    make_config(cfg_path, model_overrides=cfg["model_ov"], preprocessing_overrides=cfg["prep_ov"])
    for seed in SEEDS:
        run_name = f"{cfg['name']}_seed{seed}"
        run_dir  = reports_dir / run_name
        run_dir.mkdir(parents=True, exist_ok=True)
        cmd = (f"python -u src/train.py --config {cfg_path} "
               f"{BACKBONE_FLAGS} {cfg['flags']} "
               f"--seed {seed} --results_dir {run_dir}")
        print(f"\n=== {run_name} ===\n>> {cmd}\n")
        try:
            with open(run_dir / "training_log.txt", "w") as lf:
                proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                                        stderr=subprocess.STDOUT, text=True)
                for line in proc.stdout:
                    print(line, end="")
                    lf.write(line)
                proc.wait()
            status = "SUCCESS" if proc.returncode == 0 else f"FAILED ({proc.returncode})"
            results_log.append(f"{status}: {run_name}")
        except Exception as e:
            results_log.append(f"CRASH: {run_name}: {e}")
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f"\nGroup A done — {sum(1 for r in results_log if 'SUCCESS' in r)} succeeded")

## Group B — Medication section leakage test

**This is the most important validity check in the thesis.**

- B0: default `note_embeddings_mimic3.pkl` — medication section REMOVED before encoding (`remove_med_section=True`)
- B1: `note_embeddings_mimic3_with_meds.pkl` — medication section KEPT

**If B1 >> B0 (Δ > 0.005):** Notes are partially looking at the answer sheet.  
Thesis claim must be qualified: notes contribute X Jaccard, but Y of that is from discharge med list leakage.

**If B1 ≈ B0:** The model learns the same from full notes as from filtered notes — leakage is not the source of signal. Thesis claim stands cleanly.

In [ ]:
import subprocess, gc, torch

WITH_MEDS_PKL = "data/processed/note_embeddings_mimic3_with_meds.pkl"
import os
if not os.path.exists(WITH_MEDS_PKL):
    print(f"WARNING: {WITH_MEDS_PKL} not found.")
    print("Run the 'Group B prerequisite' cell above first, then re-run this cell.")
else:
    GROUP_B = [
        {
            "name": "B0_no_meds",
            "flags": "",
            "note": "Default: med section removed before ClinicalBERT encoding (leakage prevention ON)",
        },
        {
            "name": "B1_with_meds",
            "flags": f"--note_pkl {WITH_MEDS_PKL}",
            "note": "Med section KEPT — if >> B0, notes benefit from seeing discharge med list (leakage)",
        },
    ]

    cfg_path = "s12a_group_b.yaml"
    make_config(cfg_path, model_overrides=NOTES_MODEL_OV, preprocessing_overrides=NOTES_PREP_OV)

    for cfg in GROUP_B:
        print(f"\n{'#'*70}\n# {cfg['name']} — {cfg['note']}\n{'#'*70}")
        for seed in SEEDS:
            run_name = f"{cfg['name']}_seed{seed}"
            run_dir  = reports_dir / run_name
            run_dir.mkdir(parents=True, exist_ok=True)
            cmd = (f"python -u src/train.py --config {cfg_path} "
                   f"{BACKBONE_FLAGS} {cfg['flags']} "
                   f"--seed {seed} --results_dir {run_dir}")
            print(f"\n=== {run_name} ===\n>> {cmd}\n")
            try:
                with open(run_dir / "training_log.txt", "w") as lf:
                    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                                            stderr=subprocess.STDOUT, text=True)
                    for line in proc.stdout:
                        print(line, end="")
                        lf.write(line)
                    proc.wait()
                status = "SUCCESS" if proc.returncode == 0 else f"FAILED ({proc.returncode})"
                results_log.append(f"{status}: {run_name}")
            except Exception as e:
                results_log.append(f"CRASH: {run_name}: {e}")
            gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()

    print("Group B done.")

## Group C — Historical notes

Does including notes from prior visits improve predictions for the current visit?

- C0: current visit note only (default)
- C1: `--use_hist_notes` — blends mean-pooled history notes with current visit note

In [ ]:
import subprocess, gc, torch

GROUP_C = [
    {
        "name": "C0_no_hist_notes",
        "flags": "",
        "note": "Current visit note only (default)",
    },
    {
        "name": "C1_hist_notes",
        "flags": "--use_hist_notes",
        "note": "Prior visit notes blended in — does note history across visits help?",
    },
]

cfg_path = "s12a_group_c.yaml"
make_config(cfg_path, model_overrides=NOTES_MODEL_OV, preprocessing_overrides=NOTES_PREP_OV)

for cfg in GROUP_C:
    print(f"\n{'#'*70}\n# {cfg['name']} — {cfg['note']}\n{'#'*70}")
    for seed in SEEDS:
        run_name = f"{cfg['name']}_seed{seed}"
        run_dir  = reports_dir / run_name
        run_dir.mkdir(parents=True, exist_ok=True)
        cmd = (f"python -u src/train.py --config {cfg_path} "
               f"{BACKBONE_FLAGS} {cfg['flags']} "
               f"--seed {seed} --results_dir {run_dir}")
        print(f"\n=== {run_name} ===\n>> {cmd}\n")
        try:
            with open(run_dir / "training_log.txt", "w") as lf:
                proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                                        stderr=subprocess.STDOUT, text=True)
                for line in proc.stdout:
                    print(line, end="")
                    lf.write(line)
                proc.wait()
            status = "SUCCESS" if proc.returncode == 0 else f"FAILED ({proc.returncode})"
            results_log.append(f"{status}: {run_name}")
        except Exception as e:
            results_log.append(f"CRASH: {run_name}: {e}")
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f"\nAll groups done — {len(results_log)} runs | {sum(1 for r in results_log if 'SUCCESS' in r)} succeeded")

In [ ]:
import json, numpy as np
from pathlib import Path

reports_dir = Path("experiment_reports/sweep_12a_notes_core")

def get_metric(d, key):
    sec = d.get("test_metrics", {})
    for k in [key, key.replace("_", " "), key.title()]:
        if k in sec: return float(sec[k])
        if k in d:   return float(d[k])
    return 0.0

results = {}
for jp in sorted(reports_dir.rglob("result_*.json")):
    with open(jp) as f: d = json.load(f)
    run_name = jp.parent.name
    idx = run_name.rfind("_seed")
    if idx == -1: continue
    results.setdefault(run_name[:idx], []).append(d)

def summarize(name):
    runs = results.get(name, [])
    if not runs: return None
    jac = [get_metric(d, "jaccard") for d in runs]
    f1  = [get_metric(d, "f1")      for d in runs]
    ddi = [get_metric(d, "ddi_rate") for d in runs]
    return {"jac_mean": np.mean(jac), "jac_std": np.std(jac, ddof=1) if len(jac)>1 else 0.0,
            "f1": np.mean(f1), "ddi": np.mean(ddi), "n": len(jac)}

def row(name, label, ref_jac=None):
    s = summarize(name)
    if not s:
        print(f"  {label:<32} (missing)")
        return 0.0
    delta = f"{s['jac_mean']-ref_jac:+.4f}" if ref_jac is not None else "     —"
    print(f"  {label:<32} {s['jac_mean']:.4f}±{s['jac_std']:.4f}  {delta:>8}  F1={s['f1']:.4f}  n={s['n']}")
    return s["jac_mean"]

print("\n" + "="*75)
print("GROUP A — Core modality sweep")
print("="*75)
a0  = row("A0_naked",      "A0: naked")
a1  = row("A1_notes_only", "A1: +notes only",    a0)
a2  = row("A2_notes_copy", "A2: +notes+copy",    a0)
a3  = row("A3_notes_labs", "A3: +notes+labs",    a0)
a4  = row("A4_full",       "A4: full system",    a0)
print(f"\n  Δ_notes (A1−A0): {a1-a0:+.4f}")
print(f"  Δ_copy  (A2−A1): {a2-a1:+.4f}  (copy on top of notes)")
print(f"  Δ_labs  (A3−A1): {a3-a1:+.4f}  (labs on top of notes)")
print(f"  Δ_full  (A4−A0): {a4-a0:+.4f}  (total contribution)")

print("\n" + "="*75)
print("GROUP B — Medication section leakage test")
print("="*75)
b0 = row("B0_no_meds",   "B0: med section removed (default)", a0)
b1 = row("B1_with_meds", "B1: med section kept",              a0)
leak = b1 - b0
print(f"\n  Leakage estimate (B1−B0): {leak:+.4f}")
if leak > 0.010:
    print(f"  ⚠ SIGNIFICANT LEAKAGE: notes benefit {leak:.4f} Jaccard from discharge med list.")
    print(f"    Thesis claim must be qualified. True clean Δ_notes = {b0-a0:+.4f}")
elif leak > 0.003:
    print(f"  ~ MINOR LEAKAGE ({leak:.4f}). Mention in limitations; primary claim (B0) holds.")
else:
    print(f"  ✓ NO MEANINGFUL LEAKAGE. Med section filter does not significantly change results.")
    print(f"    Note signal is genuine clinical content, not discharge med list memorization.")

print("\n" + "="*75)
print("GROUP C — Historical notes")
print("="*75)
c0 = row("C0_no_hist_notes", "C0: current visit only", a1)
c1 = row("C1_hist_notes",    "C1: + history notes",   a1)
print(f"\n  Δ hist_notes: {c1-c0:+.4f}")

In [ ]:
import zipfile
from pathlib import Path

zip_name = "reports_sweep_12a_notes_core.zip"
rd = Path("experiment_reports/sweep_12a_notes_core")
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(rd.rglob("result_*.json")): zf.write(p, p.relative_to(rd))
    for p in sorted(rd.rglob("training_log.txt")): zf.write(p, p.relative_to(rd))
print(f"Exported → {zip_name}")